In [ ]:
!pip install -q mlflow

In [ ]:
import os, glob, re
import numpy as np, pandas as pd

SEED = 42
WORK = "/kaggle/working"
df = pd.read_parquet(glob.glob("/kaggle/input/**/ml_dataset.parquet", recursive=True)[0])
df = df.sort_values("ts", kind="mergesort").reset_index(drop=True)

# Réseau surveillé : 192.168.x.x (serveur web de l'université)
is_int = lambda s: s.astype(str).str.startswith("192.168.")
src_int, dst_int = is_int(df["src_ip"]), is_int(df["dest_ip"])
df["direction"] = np.select(
    [~src_int & dst_int, src_int & ~dst_int, src_int & dst_int],
    ["entrant", "sortant", "interne"], default="externe")

print("Alertes et taux de positifs par direction :")
print(df.groupby("direction")["label"].agg(alertes="size", positifs="sum", taux="mean").round(3).to_string())

print("\nIP sources internes (sortant / interne) :")
print(df.loc[src_int, "src_ip"].value_counts().head(5).to_string())

for d in ["sortant", "entrant"]:
    sub = df[(df["direction"] == d) & (df["label"] == 1)]
    print(f"\nSignatures des positifs [{d}] (top 8) :")
    print(sub["sig"].value_counts().head(8).to_string())
    print(f"Ports destination [{d}] :", df.loc[df["direction"] == d, "dest_port"].value_counts().head(5).to_dict())

In [ ]:
it = df[df["direction"] == "interne"]
en = df[df["direction"] == "entrant"]

print("IP destination (interne) :", it["dest_ip"].value_counts().head(5).to_dict())
print("Ports destination (interne) :", it["dest_port"].value_counts().head(5).to_dict())
print("app_proto (interne) :", it["app_proto"].value_counts(dropna=False).head(5).to_dict())
print("IP destination (entrant) :", en["dest_ip"].value_counts().head(3).to_dict())

print("\nSignatures des positifs [interne] (top 10) :")
print(it.loc[it["label"] == 1, "sig"].value_counts().head(10).to_string())
print("\nSignatures des négatifs [interne] (top 5) :")
print(it.loc[it["label"] == 0, "sig"].value_counts().head(5).to_string())

# Test du « double comptage » : pour chaque alerte interne, existe-t-il une alerte ENTRANTE
# avec la MÊME signature dans les 2 secondes qui précèdent ?
a = it[["ts", "sig", "label"]].sort_values("ts")
b = en[["ts", "sig", "src_ip"]].rename(columns={"src_ip": "src_externe"}).sort_values("ts")
m = pd.merge_asof(a, b, on="ts", by="sig", direction="backward", tolerance=pd.Timedelta(seconds=2))
print("\nAlertes internes précédées d'une alerte entrante identique (< 2 s) :",
      round(m["src_externe"].notna().mean(), 3))
print("  parmi les positifs internes :", round(m.loc[m["label"] == 1, "src_externe"].notna().mean(), 3))
print("  parmi les négatifs internes :", round(m.loc[m["label"] == 0, "src_externe"].notna().mean(), 3))

In [ ]:
import shutil, mlflow
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score

host_feats = ["hist_flows", "hist_alerted", "hist_alert_rate", "is_new_host",
              "prev1h_flows", "prev1h_alerted", "prev1h_ports"]
cat = ["proto", "app_proto", "state", "reason"]
num = ["dest_port", "pkts_toserver", "pkts_toclient", "bytes_toserver", "bytes_toclient",
       "age", "bytes_ratio", "pkts_total", "hour", "dow", "ip_prev_alerts", "ip_gap_s"] + host_feats

X = pd.get_dummies(df[num + cat], columns=cat, dummy_na=True).astype("float32").fillna(0)
X.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", c) for c in X.columns]
y = df["label"].values

t_cut = df["ts"].iloc[int(len(df) * 0.8)]          # même coupure temporelle que le notebook principal
is_tr, is_te = (df["ts"] < t_cut).values, (df["ts"] >= t_cut).values
is_in = (df["direction"] == "entrant").values

def lgbm():
    return LGBMClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, verbose=-1)

def scores(y_true, p):
    pr = (p >= 0.5).astype(int)
    return {"n_test": int(len(y_true)), "prevalence": round(float(y_true.mean()), 3),
            "auc_pr": average_precision_score(y_true, p), "roc_auc": roc_auc_score(y_true, p),
            "precision": precision_score(y_true, pr), "recall": recall_score(y_true, pr),
            "f1": f1_score(y_true, pr)}

m_all = lgbm().fit(X[is_tr], y[is_tr])
m_in = lgbm().fit(X[is_tr & is_in], y[is_tr & is_in])
p_in = m_in.predict_proba(X[is_te & is_in])[:, 1]

res = pd.DataFrame({
    "(a) tout → tout":            scores(y[is_te], m_all.predict_proba(X[is_te])[:, 1]),
    "(b) tout → entrant":         scores(y[is_te & is_in], m_all.predict_proba(X[is_te & is_in])[:, 1]),
    "(c) entrant → entrant":      scores(y[is_te & is_in], p_in),
}).T
print("Lignes d'entraînement : tout =", int(is_tr.sum()), "| entrant =", int((is_tr & is_in).sum()))
print(res.round(3).to_string())

# IC 95 % bootstrap par IP pour le nouveau modèle de référence (c)
yt = y[is_te & is_in]
codes, _ = pd.factorize(df.loc[is_te & is_in, "src_ip"])
groups = pd.Series(np.arange(len(yt))).groupby(codes).apply(np.array).values
rng, boots = np.random.RandomState(SEED), []
for _ in range(500):
    ii = np.concatenate(groups[rng.randint(0, len(groups), len(groups))])
    if yt[ii].sum():
        boots.append(average_precision_score(yt[ii], p_in[ii]))
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"\n(c) AUC-PR = {res.loc['(c) entrant → entrant', 'auc_pr']:.3f}  IC 95 % bootstrap par IP [{lo:.3f} ; {hi:.3f}]"
      f" | IP distinctes : {len(groups)}")

# MLflow : reprendre la base la plus complète disponible (celle du notebook MLP si elle est attachée)
dbs = sorted(glob.glob("/kaggle/input/**/mlflow.db", recursive=True), key=lambda p: "deep" not in p)
if dbs and not os.path.exists(f"{WORK}/mlflow.db"):
    shutil.copy(dbs[0], f"{WORK}/mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{WORK}/mlflow.db")
if mlflow.get_experiment_by_name("suricata-detection") is None:
    mlflow.create_experiment("suricata-detection", artifact_location=f"{WORK}/mlartifacts")
mlflow.set_experiment("suricata-detection")
with mlflow.start_run(run_name="lightgbm_entrant_seul"):
    mlflow.log_params({"model": "lightgbm", "config": "défaut", "perimetre": "trafic entrant (hors proxy → backend)",
                       "split": "temporal_80_20"})
    mlflow.log_metrics({k: float(v) for k, v in res.loc["(c) entrant → entrant"].items()} |
                       {"auc_pr_ci_low": lo, "auc_pr_ci_high": hi})
print("Base MLflow utilisée :", dbs[0] if dbs else "nouvelle")